# Dimensão Colaboradores

Como os dados são de testes o objetivos é criar um histórico com base na base de eventos, ou seja, quando chegar no mês que o funcionário foi demitido ele fica com aquela data e o seu registro não avança.

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from datetime import datetime, date

In [0]:
def pick_months(start: date, end: date):
    """ Retornar os meses entre duas datas """
    i, m = start.year, start.month 
    
    while (i, m) <= (end.year, end.month):
        yield date(i, m, 1)
        m += 1
        if m > 12:
            m = 1
            i += 1

Configurações

In [0]:
print('⚒️ Iniciando configuração')

DATA_INICIAL = date(2013, 1, 1)
DATA_FINAL = datetime.now().replace(day=1)
TARGET_TABLE_PATH = 'projetos.gold.dim_employees'

print(f'Range definido para {DATA_INICIAL} <-> {DATA_FINAL.strftime('%Y-%m-%d')}')

Leitura das bases

In [0]:
print("🔍 Lendo as tabelas do processo")

events = spark.table('projetos.silver.tab_eventos_colaboradores').drop("source_file", "file_modification_time") 
workers = spark.table('projetos.silver.tab_workers').drop("source_file", "file_modification_time")

In [0]:

for i in pick_months(DATA_INICIAL, DATA_FINAL):
    print(f'⏳ Processando: {i}')
    workers = workers.withColumn('dt_registro', F.lit(i))

    events_sel = events.select(
        F.col('Worker_ID').alias('Worker_ID_s'), 'event_reason', 'effective_date'
    )

    df = workers.join(events_sel, (workers.Worker_ID == events_sel.Worker_ID_s) & (workers.dt_registro == F.to_date(F.date_trunc('month', events_sel.effective_date))), how="left")
    
    # Descobrir se o funcionário está ativo

    df = df.withColumn('flag_contratado', F.when(F.col('event_reason').like('%Terminate%'), 0).otherwise(1))
    df = df.withColumn('dt_registro',     F.when(F.col('event_reason').like('%Terminate%'), 
                                                 F.to_date(F.date_trunc('month', events_sel.effective_date))).otherwise(F.col('dt_registro')))
    
    df = df.withColumn('update_at', F.current_timestamp()).drop('event_reason', 'effective_date', 'Worker_ID_s')
    df = df.withColumn('OP', F.lit('I'))

    df = df.select('dt_registro', *[c for c in df.columns if c != 'dt_registro'])

    print(f'✅ Funcionários ativos {df.filter(F.col('flag_contratado') == 1).count()} \n❌ Funcionários inativos: {df.filter(F.col('flag_contratado') == 0).count()}')
    
    # Save

    if spark.catalog.tableExists(TARGET_TABLE_PATH):
        target_table = DeltaTable.forName(spark, TARGET_TABLE_PATH)
        
        print(f'⏳ Tabela existe! {TARGET_TABLE_PATH}')
        update_set = {c: F.col(f'source.{c}') for c in df.columns}
        update_set['OP'] = F.lit('U')
        update_set['update_at'] = F.current_timestamp()

        target_table.alias('target').merge(df.alias('source'), condition="target.Worker_ID = source.Worker_ID and target.dt_registro = source.dt_registro") \
            .whenMatchedUpdate(set=update_set)\
            .whenNotMatchedInsertAll()\
            .execute()
    else:
        print('⏳ Tabela não existe! ')
        df.write.mode('append').partitionBy('dt_registro').saveAsTable(TARGET_TABLE_PATH)